In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('customer_data.csv')
print(f"Data loaded: {df.shape[0]} customers, {df.shape[1]} columns")
df.head()

Data loaded: 50 customers, 10 columns


,customer_id,customer_name,industry,contract_value,logins_last_30_days,feature_adoption_rate,open_tickets,days_since_last_login,contract_renewal_days,ai_acceptance_rate
0,C001,Jordan-Escobar,Telecom,150000,27,0.59,6,48,31,0.66
1,C002,Ramirez-Brown,Railways,150000,6,0.28,7,16,247,0.28
2,C003,"Bray, Austin and Douglas",Water Utility,150000,8,0.95,0,3,167,0.69
3,C004,Moyer and Sons,Railways,200000,7,0.64,5,20,47,0.99
4,C005,Wilson LLC,Railways,100000,11,0.73,7,24,239,0.31


In [9]:
def score_logins(logins):
    # max possible logins = 30, normalize to 0-100
    return round(min(logins / 30 * 100, 100), 2)

def score_feature_adoption(rate):
    # already between 0 and 1, scale to 0-100
    return round(rate * 100, 2)

def score_open_tickets(tickets):
    # more tickets = lower score
    # 0 tickets = 100, 10 tickets = 0
    return round(max(100 - (tickets * 10), 0), 2)

def score_days_since_login(days):
    # fewer days = higher score
    # 0 days = 100, 60+ days = 0
    return round(max(100 - (days / 60 * 100), 0), 2)

def score_ai_acceptance(rate):
    # already between 0 and 1, scale to 0-100
    return round(rate * 100, 2)

# test each function manually
print("Login score (15 logins)      :", score_logins(15))
print("Adoption score (0.6 rate)    :", score_feature_adoption(0.6))
print("Ticket score (3 tickets)     :", score_open_tickets(3))
print("Last login score (20 days)   :", score_days_since_login(20))
print("AI acceptance score (0.8)    :", score_ai_acceptance(0.8))

Login score (15 logins)      : 50.0
Adoption score (0.6 rate)    : 60.0
Ticket score (3 tickets)     : 70
Last login score (20 days)   : 66.67
AI acceptance score (0.8)    : 80.0


In [10]:
df['login_score']     = df['logins_last_30_days'].apply(score_logins)
df['adoption_score']  = df['feature_adoption_rate'].apply(score_feature_adoption)
df['ticket_score']    = df['open_tickets'].apply(score_open_tickets)
df['recency_score']   = df['days_since_last_login'].apply(score_days_since_login)
df['ai_score']        = df['ai_acceptance_rate'].apply(score_ai_acceptance)

print("Individual scores added")
df[['customer_name', 'login_score', 'adoption_score', 
    'ticket_score', 'recency_score', 'ai_score']].head(10)

Individual scores added


,customer_name,login_score,adoption_score,ticket_score,recency_score,ai_score
0,Jordan-Escobar,90.00,59.0,40,20.00,66.0
1,Ramirez-Brown,20.00,28.0,30,73.33,28.0
2,"Bray, Austin and Douglas",26.67,95.0,100,95.00,69.0
3,Moyer and Sons,23.33,64.0,50,66.67,99.0
4,Wilson LLC,36.67,73.0,30,60.00,31.0
5,Rivera-Kramer,3.33,89.0,60,10.00,61.0
6,Martin LLC,0.00,66.0,70,6.67,90.0
7,Blair Group,50.00,37.0,90,45.00,79.0
8,Garcia LLC,73.33,19.0,50,60.00,76.0
9,"Armstrong, Torres and Hernandez",73.33,51.0,50,13.33,76.0


In [11]:
WEIGHTS = {
    'login_score'    : 0.25,
    'adoption_score' : 0.25,
    'ticket_score'   : 0.20,
    'recency_score'  : 0.15,
    'ai_score'       : 0.15
}

df['health_score'] = round(
    df['login_score']    * WEIGHTS['login_score']    +
    df['adoption_score'] * WEIGHTS['adoption_score'] +
    df['ticket_score']   * WEIGHTS['ticket_score']   +
    df['recency_score']  * WEIGHTS['recency_score']  +
    df['ai_score']       * WEIGHTS['ai_score'], 2
)

print(f"Health scores calculated")
print(f"Min  : {df['health_score'].min()}")
print(f"Max  : {df['health_score'].max()}")
print(f"Mean : {df['health_score'].mean().round(2)}")

Health scores calculated
Min  : 33.2
Max  : 75.65
Mean : 51.62


In [12]:
def classify_risk(score):
    if score >= 75:
        return 'Healthy'
    elif score >= 50:
        return 'Needs Attention'
    else:
        return 'At Risk'

df['risk_level'] = df['health_score'].apply(classify_risk)

print("=== Risk Distribution ===")
print(df['risk_level'].value_counts())
df[['customer_name', 'health_score', 'risk_level']].sort_values(
    'health_score').head(10)

=== Risk Distribution ===
risk_level
At Risk            28
Needs Attention    20
Healthy             2
Name: count, dtype: int64


,customer_name,health_score,risk_level
1,Ramirez-Brown,33.20,At Risk
30,"Mcdaniel, Harris and Frazier",35.20,At Risk
19,"Evans, Stephens and Christensen",38.05,At Risk
25,Wright and Sons,38.70,At Risk
32,Pope PLC,38.72,At Risk
34,"Walker, Rodriguez and Duffy",41.32,At Risk
21,Sanders and Sons,41.93,At Risk
37,"Santos, Tran and Baker",42.55,At Risk
20,Martinez Ltd,43.82,At Risk
39,Thomas and Sons,44.05,At Risk


In [13]:
def recommend_action(row):
    if row['risk_level'] == 'Healthy':
        return 'No action needed — schedule quarterly check-in'
    
    actions = []
    
    if row['logins_last_30_days'] < 5:
        actions.append('Schedule re-engagement call')
    
    if row['feature_adoption_rate'] < 0.4:
        actions.append('Offer product training session')
    
    if row['open_tickets'] > 5:
        actions.append('Escalate open tickets to support lead')
    
    if row['days_since_last_login'] > 30:
        actions.append('Send personalised check-in email')
    
    if row['ai_acceptance_rate'] < 0.4:
        actions.append('Review AI recommendation quality with R&D')
    
    if row['contract_renewal_days'] < 60:
        actions.append('Urgent — flag for renewal discussion')

    return ' | '.join(actions) if actions else 'Monitor closely'

df['recommended_action'] = df.apply(recommend_action, axis=1)

print("Recommendations generated")
df[['customer_name', 'risk_level', 'recommended_action']].head(10)

Recommendations generated


,customer_name,risk_level,recommended_action
0,Jordan-Escobar,Needs Attention,Escalate open tickets to support lead | Send p...
1,Ramirez-Brown,At Risk,Offer product training session | Escalate open...
2,"Bray, Austin and Douglas",Healthy,No action needed — schedule quarterly check-in
3,Moyer and Sons,Needs Attention,Urgent — flag for renewal discussion
4,Wilson LLC,At Risk,Escalate open tickets to support lead | Review...
5,Rivera-Kramer,At Risk,Schedule re-engagement call | Send personalise...
6,Martin LLC,At Risk,Schedule re-engagement call | Send personalise...
7,Blair Group,Needs Attention,Offer product training session | Send personal...
8,Garcia LLC,Needs Attention,Offer product training session | Urgent — flag...
9,"Armstrong, Torres and Hernandez",Needs Attention,Send personalised check-in email


In [14]:
fig = px.histogram(
    df,
    x='health_score',
    color='risk_level',
    nbins=20,
    title='Distribution of Customer Health Scores',
    color_discrete_map={
        'Healthy'         : '#2ecc71',
        'Needs Attention' : '#f39c12',
        'At Risk'         : '#e74c3c'
    },
    labels={'health_score': 'Health Score', 'count': 'Number of Customers'}
)
fig.show()

In [15]:
fig = px.scatter(
    df,
    x='logins_last_30_days',
    y='health_score',
    color='risk_level',
    size='contract_value',
    hover_data=['customer_name', 'recommended_action'],
    title='Login Frequency vs Health Score (bubble size = contract value)',
    color_discrete_map={
        'Healthy'         : '#2ecc71',
        'Needs Attention' : '#f39c12',
        'At Risk'         : '#e74c3c'
    }
)
fig.show()


In [16]:
industry_risk = df.groupby(
    ['industry', 'risk_level']).size().reset_index(name='count')

fig = px.bar(
    industry_risk,
    x='industry',
    y='count',
    color='risk_level',
    barmode='group',
    title='Risk Level Breakdown by Industry',
    color_discrete_map={
        'Healthy'         : '#2ecc71',
        'Needs Attention' : '#f39c12',
        'At Risk'         : '#e74c3c'
    }
)
fig.show()

In [17]:
df.to_csv('customer_data_scored.csv', index=False)

print("Scored dataset saved as customer_data_scored.csv")
print(f"Total columns now: {df.shape[1]}")
print(f"New columns added: health_score, risk_level, recommended_action")
print(f"\nFinal risk summary:")
print(df['risk_level'].value_counts())

Scored dataset saved as customer_data_scored.csv
Total columns now: 18
New columns added: health_score, risk_level, recommended_action

Final risk summary:
risk_level
At Risk            28
Needs Attention    20
Healthy             2
Name: count, dtype: int64
